In [1]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [2]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

In [3]:
code = 'SPRMF'
market = ''
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=code,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [4]:
def stock_prices_and_material_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        df_sp: pd.DataFrame | None = None,
        df_mat1: pd.DataFrame | None = None,
        df_mat2: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=None,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # mat1価格を統合
    if df_mat1 is not None:
        df_mat1_tmp = df_mat1.copy() if df_mat1 is not None else pd.DataFrame()
        if "date" not in df_mat1_tmp.columns:
            df_mat1_tmp = df_mat1_tmp.reset_index()
        df_mat1_tmp["date"] = pd.to_datetime(df_mat1_tmp["date"])
        df_mat1_tmp = df_mat1_tmp.set_index("date")
        df_mat1_tmp = df_mat1_tmp.loc[start:end]

    # mat2価格を統合
    if df_mat2 is not None:
        df_mat2_tmp = df_mat2.copy() if df_mat2 is not None else pd.DataFrame()
        if "date" not in df_mat2_tmp.columns:
            df_mat2_tmp = df_mat2_tmp.reset_index()
        df_mat2_tmp["date"] = pd.to_datetime(df_mat2_tmp["date"])
        df_mat2_tmp = df_mat2_tmp.set_index("date")
        df_mat2_tmp = df_mat2_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_mat1 is not None:
        df["MA5_MAT1"] = df_mat1_tmp["ma5"].reindex(df.index)
        df["MA25_MAT1"] = df_mat1_tmp["ma25"].reindex(df.index)
    if df_mat2 is not None:
        df["MA5_MAT2"] = df_mat2_tmp["ma5"].reindex(df.index)
        df["MA25_MAT2"] = df_mat2_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT1（右軸） ---
    if df_mat1 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT1"],
                name="MAT1_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT1"],
                name="MAT1_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT2（左軸） ---
    if df_mat2 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT2"],
                name="MAT2_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT2"],
                name="MAT2_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [ ]:
name = ""
start = dt.datetime(2025, 12, 11).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_material_prices(
    code=code,
    name=name,
    start=start,
    end=end,
    df_sp=None,
    df_mat1=None,
    df_mat2=None
)
fig.show()

取得件数: 171


In [8]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://spartanmetals.com/wp-content/uploads/2025/12/Spartan-Metals-FS-Q3-2025-Sept-30-25-Final-11-28-25-Final-Sedar.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/Spartan-Metals-FS-Q3-2025-Sept-30-25-Final-11-28-25-Final-Sedar.pdf.md


'/workspace/data/Spartan-Metals-FS-Q3-2025-Sept-30-25-Final-11-28-25-Final-Sedar.pdf.md'

In [9]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://spartanmetals.com/financial-reports/#:~:text=Financial%20Statements-,MD%26A,-Q4",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/financial-reports.md


'/workspace/data/financial-reports.md'